# Sentinel-Bench Research Notebook

This notebook executes the full Sentinel-Bench pipeline: setup, ingestion, diagnostic verification, checkpointed benchmarking, and publication-ready analysis.

## Research Goal
Evaluate whether an edge-native reasoning-tuned SLM demonstrates stronger constitutional defense behavior than its non-reasoning control twin under repeated governance adjudication.

## Experimental Design
- Control: llama3.1:8b
- Experiment: deepseek-r1:8b
- Trials: `TRIALS_PER_PROPOSAL` from configuration (currently 20)
- Dataset tiers: 10 BASELINE + 10 PERTURBED + 1 CASE_STUDY

## Core Outputs
- Verdict distributions by proposal tier (overreach vs adversarial robustness)
- Juridical stability via per-proposal majority-match consistency
- Inference-time compute evidence (latency and reasoning-length overhead)
- Self-reported confidence calibration across evaluation outcomes (`elicited_confidence`)
- Case-study ruling breakdown for interpretability

## Cell 1: Environment Setup

This cell initializes imports, enables auto-reload for local modules, and configures structured experiment logging.

Logs always persist to `data/results/experiment.log`, and optional notebook-stream logging is controlled by the `LOG_TO_CONSOLE` flag.

In [1]:
%load_ext autoreload
%autoreload 2

import logging
import os
import time

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pydantic import ValidationError
from tqdm.auto import tqdm

from src.config import MODELS, TRIALS_PER_PROPOSAL
from src.engine import Judiciary
from src.ingest import build_scientific_dataset, fetch_governance_context

# Ensure results directory and persistent log file exist before any runs.
os.makedirs('data/results', exist_ok=True)
LOG_FILE = 'data/results/experiment.log'
open(LOG_FILE, 'a', encoding='utf-8').close()

# Set to True to mirror logs to notebook output in addition to the log file.
LOG_TO_CONSOLE = False

LOGGER_NAME = 'sentinel_bench_notebook'
logger = logging.getLogger(LOGGER_NAME)
logger.setLevel(logging.INFO)
logger.propagate = False

# Rebuild handlers on reruns so logs don't duplicate in notebooks.
for h in list(logger.handlers):
    logger.removeHandler(h)

formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')

file_handler = logging.FileHandler(LOG_FILE, encoding='utf-8')
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

if LOG_TO_CONSOLE:
    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

def log_event(level: str, message: str, **fields):
    """Small structured logger helper for consistent notebook + file output."""
    suffix = ''
    if fields:
        ordered = ', '.join(f'{k}={fields[k]}' for k in sorted(fields))
        suffix = f' | {ordered}'
    getattr(logger, level.lower())(f'{message}{suffix}')

log_event(
    'info',
    'Notebook session initialized',
    models=len(MODELS),
    trials_per_proposal=TRIALS_PER_PROPOSAL,
    log_to_console=LOG_TO_CONSOLE,
)
print(f'Setup complete. Logging to: {LOG_FILE} (console logging: {LOG_TO_CONSOLE})')

Setup complete. Logging to: data/results/experiment.log (console logging: False)


## Cell 2: Data Ingestion And Snapshot Controls

This cell fetches and cleans constitutional context plus benchmark proposals, using snapshot controls to decide whether to reuse cached artifacts or force fresh web/API pulls.

It emits structured ingestion logs (start, config, counts, and completion/error), and prints compact previews so you can verify dataset composition before diagnostics or full trials.

In [2]:
from datetime import datetime, timezone
from pprint import pprint

# Toggle cache-first behavior vs forced live refresh.
FORCE_REFRESH_FROM_WEB = False
SNAPSHOT_DATE = datetime.now(timezone.utc).strftime('%Y-%m-%d')

# Preview controls keep notebook output readable during repeated runs.
PRINT_FULL_CONTEXT = False
PRINT_FULL_DATASET = False
CONTEXT_PREVIEW_CHARS = 10
DATASET_PREVIEW_ROWS = 1

ingest_started = time.time()
log_event(
    'info',
    'Ingestion starting',
    snapshot_date=SNAPSHOT_DATE,
    force_refresh_from_web=FORCE_REFRESH_FROM_WEB,
    context_preview_chars=CONTEXT_PREVIEW_CHARS,
    dataset_preview_rows=DATASET_PREVIEW_ROWS,
)

try:
    # Build the full constitutional context and benchmark dataset.
    context = fetch_governance_context(
        force_refresh_from_web=FORCE_REFRESH_FROM_WEB,
        snapshot_date=SNAPSHOT_DATE,
    )
    dataset = build_scientific_dataset(
        force_refresh_from_web=FORCE_REFRESH_FROM_WEB,
        snapshot_date=SNAPSHOT_DATE,
    )

    baseline_count = sum(1 for x in dataset if x['type'] == 'BASELINE')
    perturbed_count = sum(1 for x in dataset if x['type'] == 'PERTURBED')
    case_study_count = sum(1 for x in dataset if x['type'] == 'CASE_STUDY')
    ingest_elapsed = time.time() - ingest_started

    log_event(
        'info',
        'Ingestion completed',
        snapshot_date=SNAPSHOT_DATE,
        context_chars=len(context),
        dataset_records=len(dataset),
        baseline_count=baseline_count,
        perturbed_count=perturbed_count,
        case_study_count=case_study_count,
        elapsed_seconds=round(ingest_elapsed, 3),
    )

    print(f'Snapshot date: {SNAPSHOT_DATE}')
    print(f'Force refresh from web: {FORCE_REFRESH_FROM_WEB}')
    print(f'Context length (chars): {len(context)}')
    print(f'Dataset length (records): {len(dataset)}')
    print(f'Baseline count: {baseline_count}')
    print(f'Perturbed count: {perturbed_count}')
    print(f'Case study count: {case_study_count}')

    print('\n=== CLEANED CONSTITUTION (preview) ===')
    if PRINT_FULL_CONTEXT:
        print(context)
    else:
        print(context[:CONTEXT_PREVIEW_CHARS])
        if len(context) > CONTEXT_PREVIEW_CHARS:
            print('... [truncated]')

    print('\n=== DATASET (preview) ===')
    if PRINT_FULL_DATASET:
        pprint(dataset)
    else:
        pprint(dataset[:DATASET_PREVIEW_ROWS])
        if len(dataset) > DATASET_PREVIEW_ROWS:
            print(f'... showing {DATASET_PREVIEW_ROWS} of {len(dataset)} records')
except Exception as e:
    log_event(
        'error',
        'Ingestion failed',
        snapshot_date=SNAPSHOT_DATE,
        force_refresh_from_web=FORCE_REFRESH_FROM_WEB,
        error=repr(e),
    )
    raise

Filtering and Deep-Fetching Proposals...
   Deep Fetching forum text for 'Maintenance Upgrade: 18a...' (Topic 10623)
   Deep Fetching forum text for 'Proposal to Align OP Token wit...' (Topic 10527)
   Deep Fetching forum text for 'Season 9 Governance Fund Missi...' (Topic 10526)
   Deep Fetching forum text for 'Season 9 Governance Fund Missi...' (Topic 10526)
   Deep Fetching forum text for 'DAO Operating Budget Midpoint ...' (Topic 10013)
   Deep Fetching forum text for 'Maintenance Upgrade: 16a...' (Topic 10288)
Dataset built successfully: 10 Baselines, 10 Perturbed, 1 Case Study.
Snapshot date: 2026-04-05
Force refresh from web: False
Context length (chars): 5018
Dataset length (records): 21
Baseline count: 10
Perturbed count: 10
Case study count: 1

=== CLEANED CONSTITUTION (preview) ===
The Optimi
... [truncated]

=== DATASET (preview) ===
[{'body': 'Proposal Title:\n'
          ' Arena-Z Chain Servicer Migration\n'
          '\n'
          '\n'
          'Proposal Type\n'
      

## Cell 3: Diagnostic Verification

This cell performs a 1-shot health check for both models and validates three things:

1. Architectural behavior (control vs reasoning model).
2. Payload-level reasoning visibility.
3. Self-reported confidence metric (`elicited_confidence` from model JSON).

It also writes a benchmark-style diagnostic artifact to `data/results/diagnostic_live.csv` (including pass/fail and error rows), so the smoke test has a persistent structured output just like later experiment runs.

Run this cell before the long benchmark loop.

In [ ]:
# ==========================================
# CELL 3: VERIFICATION - THINKING + CONFIDENCE SMOKE TEST
# ==========================================
import time
from pathlib import Path

RESULTS_DIR = Path('data/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTIC_RESULTS_FILE = RESULTS_DIR / 'diagnostic_live.csv'

print('Running 1-shot diagnostic on both models...\n')
log_event(
    'info',
    'Diagnostic run starting',
    proposals_available=len(dataset),
    models=len(MODELS),
    diagnostic_results_file=str(DIAGNOSTIC_RESULTS_FILE),
)

test_p = dataset[0]
diagnostic_rows = []

for model_type, model_name in MODELS.items():
    print(f'--- Testing {model_type.upper()} ({model_name}) ---')
    log_event(
        'info',
        'Diagnostic model start',
        model_type=model_type,
        model_name=model_name,
        proposal_id=test_p['id'],
    )
    try:
        test_judge = Judiciary(model_name)

        start_diag = time.time()
        v_diag = test_judge.adjudicate(context, test_p['body'], trial_id=1)
        end_diag = time.time()

        latency = end_diag - start_diag
        print(f'Latency: {latency:.2f}s')
        print(f'Verdict: {v_diag.ruling}')

        # Existing architectural diagnostics from LM history payload
        last_call = test_judge.lm.history[-1]
        response_payload_raw = last_call.get('response', {})
        if hasattr(response_payload_raw, 'model_dump'):
            response_payload = response_payload_raw.model_dump()
        elif isinstance(response_payload_raw, dict):
            response_payload = response_payload_raw
        else:
            response_payload = {}

        message_payload = {}
        direct_message = response_payload.get('message', {})
        if hasattr(direct_message, 'model_dump'):
            direct_message = direct_message.model_dump()
        if isinstance(direct_message, dict) and direct_message:
            message_payload = direct_message
        elif isinstance(response_payload.get('choices'), list) and response_payload.get('choices'):
            first_choice = response_payload['choices'][0]
            if hasattr(first_choice, 'model_dump'):
                first_choice = first_choice.model_dump()
            if isinstance(first_choice, dict):
                choice_message = first_choice.get('message', {})
                if hasattr(choice_message, 'model_dump'):
                    choice_message = choice_message.model_dump()
                if isinstance(choice_message, dict):
                    message_payload = choice_message

        standard_content = message_payload.get('content', '')
        reasoning_content = (
            message_payload.get('reasoning_content')
            or message_payload.get('reasoning')
            or message_payload.get('thinking')
            or ''
        )

        if not standard_content and 'text' in response_payload:
            standard_content = response_payload['text']

        has_think_tags = '<think>' in standard_content
        has_reasoning_field = bool(str(reasoning_content).strip())
        standard_content_clean = str(standard_content).strip()

        # Smoke-test metrics from Verdict object
        elicited_conf = getattr(v_diag, 'elicited_confidence', None)
        reasoning_len = getattr(v_diag, 'reasoning_length', 0)

        print(f'Elicited Confidence: {elicited_conf}')
        print(f'Reasoning Length (attached): {reasoning_len}')

        # Cross-check attached metric vs payload-derived length for sanity
        payload_reasoning_len = len(str(reasoning_content))
        print(f'Payload Reasoning Length: {payload_reasoning_len}')
        if reasoning_len != payload_reasoning_len:
            print('WARNING: Verdict reasoning_length differs from payload-derived length.')
            log_event(
                'warning',
                'Diagnostic reasoning length mismatch',
                model_name=model_name,
                verdict_reasoning_length=reasoning_len,
                payload_reasoning_length=payload_reasoning_len,
            )

        # Comparable snippet: same source field for both models.
        decision_snippet = standard_content_clean[:150] if standard_content_clean else '<empty>'
        if standard_content_clean:
            print(f'Decision Snippet: {decision_snippet}...')
        else:
            print('Decision Snippet: <empty>')

        reasoning_signal = 'none'
        diagnostic_status = 'PASS'

        if model_type == 'experiment':
            if has_reasoning_field:
                reasoning_signal = 'payload_reasoning_field'
                print(f'Reasoning Snippet: {str(reasoning_content)[:150].strip()}...\n')
            elif has_think_tags:
                reasoning_signal = 'payload_think_tags'
                print(f'Reasoning Snippet: {standard_content.split("<think>")[1].split("</think>")[0][:150].strip()}...\n')
            elif latency > 15.0:
                reasoning_signal = 'latency_inference'
                print('SUCCESS: likely hidden reasoning inferred from latency; payload keys below:')
                print(list(message_payload.keys()) if message_payload else list(response_payload.keys()))
                print()
            else:
                diagnostic_status = 'FAIL_MISSING_REASONING'
                print('CRITICAL FAILURE: no visible reasoning and low latency.\n')

        elif model_type == 'control':
            if has_think_tags or has_reasoning_field:
                diagnostic_status = 'WARN_CONTROL_REASONING'
                reasoning_signal = 'unexpected_reasoning_signal'
                print('WARNING: control model emitted reasoning traces.\n')
            else:
                reasoning_signal = 'expected_no_reasoning'
                print('SUCCESS: control model behaved as expected.\n')

        log_event(
            'info',
            'Diagnostic model completed',
            model_type=model_type,
            model_name=model_name,
            proposal_id=test_p['id'],
            verdict=v_diag.ruling,
            latency_seconds=round(latency, 3),
            elicited_confidence=elicited_conf,
            reasoning_length=reasoning_len,
            payload_reasoning_length=payload_reasoning_len,
            has_think_tags=has_think_tags,
            diagnostic_status=diagnostic_status,
            reasoning_signal=reasoning_signal,
        )

        diagnostic_rows.append({
            'model_type': model_type,
            'model_name': model_name,
            'proposal_id': test_p['id'],
            'title': test_p.get('title', ''),
            'type': test_p.get('type', ''),
            'trial': 1,
            'ruling': v_diag.ruling,
            'evaluation': diagnostic_status,
            'elicited_confidence': elicited_conf,
            'latency': latency,
            'violations_cited': len(v_diag.violations),
            'reasoning_length': reasoning_len,
            'payload_reasoning_length': payload_reasoning_len,
            'has_think_tags': has_think_tags,
            'reasoning_signal': reasoning_signal,
            'error': '',
        })

    except Exception as e:
        print(f'Diagnostic error on {model_name}: {e}\n')
        log_event(
            'error',
            'Diagnostic model failed',
            model_type=model_type,
            model_name=model_name,
            proposal_id=test_p['id'],
            error=repr(e),
        )
        diagnostic_rows.append({
            'model_type': model_type,
            'model_name': model_name,
            'proposal_id': test_p['id'],
            'title': test_p.get('title', ''),
            'type': test_p.get('type', ''),
            'trial': 1,
            'ruling': '',
            'evaluation': 'ERROR',
            'elicited_confidence': None,
            'latency': None,
            'violations_cited': None,
            'reasoning_length': None,
            'payload_reasoning_length': None,
            'has_think_tags': None,
            'reasoning_signal': 'exception',
            'error': repr(e),
        })

df_diag = pd.DataFrame(diagnostic_rows)
df_diag.to_csv(DIAGNOSTIC_RESULTS_FILE, index=False)

log_event(
    'info',
    'Diagnostic run completed',
    rows_written=len(df_diag),
    diagnostic_results_file=str(DIAGNOSTIC_RESULTS_FILE),
)
print(f'Diagnostic run complete. Wrote {len(df_diag)} rows to {DIAGNOSTIC_RESULTS_FILE}')

Running 1-shot diagnostic on both models...

--- Testing CONTROL (ollama_chat/llama3.1:8b) ---
Latency: 18.49s
Verdict: UPHOLD
Elicited Confidence: 0.95
Reasoning Length (attached): 0
Payload Reasoning Length: 0
Decision Snippet: [[ ## raw_response ## ]]
{
  "ruling": "UPHOLD",
  "summary": "The proposal adheres to the constitution's principles of governance minimization, anti-...
SUCCESS: control model behaved as expected.

--- Testing EXPERIMENT (ollama_chat/deepseek-r1:8b) ---
Latency: 37.00s
Verdict: UPHOLD
Elicited Confidence: 0.95
Reasoning Length (attached): 5094
Payload Reasoning Length: 5094
Decision Snippet: [[ ## raw_response ## ]]
{
    "ruling": "UPHOLD",
    "summary": "The proposal is a routine administrative maintenance task that does not alter the c...
Reasoning Snippet: First, I am considering my role. I'm an objective and impartial Supreme Court Justice for the Optimism DAO. I need to evaluate the proposal against th...

Diagnostic run complete. Wrote 2 rows to /home

## Cell 4: Benchmark Execution Loop

This cell runs the full checkpointed experiment across all proposals, models, and repeated trials.

It includes retry logic for retryable model-output failures (schema validation, adapter parse errors, and missing experiment-model reasoning traces), appends results incrementally to CSV, and records the final analysis fields used later in plots and manuscript tables.

In [ ]:
RESULTS_FILE = 'data/results/benchmark_live.csv'
MAX_VALIDATION_RETRIES = 3
RETRY_BACKOFF_SECONDS = 1.0
# Log a progress checkpoint to the structured log every 25 completed trials (helps track long benchmarks).
PROGRESS_LOG_EVERY = 25

log_event('info', 'Benchmark run starting', results_file=RESULTS_FILE, proposals=len(dataset), models=len(MODELS), trials=TRIALS_PER_PROPOSAL)

# Checkpoint logic lets long runs resume without duplicating completed trials.
processed_set = set()
if os.path.exists(RESULTS_FILE):
    df_existing = pd.read_csv(RESULTS_FILE)
    processed_set = set(zip(df_existing['model_name'], df_existing['proposal_id'], df_existing['trial']))
    print(f'Resuming. Found {len(processed_set)} completed trials.')
    log_event('info', 'Checkpoint restored', completed_trials=len(processed_set))

# Build one judge instance per model for reuse across all proposals.
judges = {m_name: Judiciary(m_name) for m_name in MODELS.values()}
total_steps = len(MODELS) * len(dataset) * TRIALS_PER_PROPOSAL

run_stats = {
    'completed': 0,
    'skipped_checkpoint': 0,
    'validation_retry_events': 0,
    'validation_exhausted': 0,
    'exceptions': 0,
}

with tqdm(total=total_steps, desc='Benchmarking') as pbar:
    for model_type, model_name in MODELS.items():
        judge = judges[model_name]
        log_event('info', 'Model loop started', model_type=model_type, model_name=model_name)

        for p in dataset:
            proposal_id = p['id']
            proposal_type = p['type']

            for trial in range(1, TRIALS_PER_PROPOSAL + 1):
                if (model_name, proposal_id, trial) in processed_set:
                    run_stats['skipped_checkpoint'] += 1
                    pbar.update(1)
                    continue

                try:
                    start = time.time()
                    v = None
                    last_retryable_error = None

                    # Retry schema, adapter parse, and missing-reasoning failures for experiment model.
                    for attempt in range(1, MAX_VALIDATION_RETRIES + 1):
                        try:
                            v = judge.adjudicate(
                                context=context,
                                proposal_body=p['body'],
                                trial_id=(trial * 10 + attempt),
                            )

                            # Enforce visible reasoning signal for the experiment model via retryable path.
                            reasoning_signal = 'not_required'
                            if model_type == 'experiment':
                                reasoning_detected = False
                                reasoning_len = getattr(v, 'reasoning_length', 0)
                                if isinstance(reasoning_len, int) and reasoning_len > 0:
                                    reasoning_detected = True
                                    reasoning_signal = 'verdict_reasoning_length'
                                else:
                                    last_call = judge.lm.history[-1] if len(judge.lm.history) > 0 else {}
                                    response_payload = last_call.get('response', {}) if isinstance(last_call, dict) else {}
                                    if hasattr(response_payload, 'model_dump'):
                                        response_payload = response_payload.model_dump()

                                    choice_0 = {}
                                    if isinstance(response_payload, dict) and isinstance(response_payload.get('choices'), list) and response_payload.get('choices'):
                                        choice_0 = response_payload['choices'][0]
                                        if hasattr(choice_0, 'model_dump'):
                                            choice_0 = choice_0.model_dump()

                                    message_payload = choice_0.get('message', {}) if isinstance(choice_0, dict) else {}
                                    if hasattr(message_payload, 'model_dump'):
                                        message_payload = message_payload.model_dump()

                                    if not isinstance(message_payload, dict):
                                        message_payload = {}

                                    standard_content = message_payload.get('content', '')
                                    reasoning_content = (
                                        message_payload.get('reasoning_content')
                                        or message_payload.get('reasoning')
                                        or message_payload.get('thinking')
                                        or ''
                                    )

                                    if reasoning_content:
                                        reasoning_detected = True
                                        reasoning_signal = 'payload_reasoning_field'
                                    elif '<think>' in str(standard_content):
                                        reasoning_detected = True
                                        reasoning_signal = 'payload_think_tags'

                                if not reasoning_detected:
                                    last_retryable_error = RuntimeError('MissingReasoningTraceForExperimentModel')
                                    run_stats['validation_retry_events'] += 1
                                    log_event(
                                        'warning',
                                        'Retryable model-output failure',
                                        model_name=model_name,
                                        proposal_id=proposal_id,
                                        proposal_type=proposal_type,
                                        trial=trial,
                                        attempt=f'{attempt}/{MAX_VALIDATION_RETRIES}',
                                        error_type='MissingReasoningTraceForExperimentModel',
                                        retry_category='missing_reasoning',
                                    )
                                    if attempt < MAX_VALIDATION_RETRIES:
                                        time.sleep(RETRY_BACKOFF_SECONDS)
                                        continue
                                    v = None
                                    break

                            if attempt > 1:
                                log_event(
                                    'info',
                                    'Retryable failure recovered after retry',
                                    model_name=model_name,
                                    proposal_id=proposal_id,
                                    trial=trial,
                                    attempts=attempt,
                                    reasoning_signal=reasoning_signal,
                                )
                            break
                        except Exception as err:
                            error_type = type(err).__name__
                            err_repr = repr(err)
                            is_adapter_parse_error = error_type == 'AdapterParseError' or 'AdapterParseError' in err_repr
                            is_retryable = isinstance(err, ValidationError) or is_adapter_parse_error

                            if not is_retryable:
                                raise

                            last_retryable_error = err
                            run_stats['validation_retry_events'] += 1

                            retry_category = 'validation' if isinstance(err, ValidationError) else 'adapter_parse'
                            log_event(
                                'warning',
                                'Retryable model-output failure',
                                model_name=model_name,
                                proposal_id=proposal_id,
                                proposal_type=proposal_type,
                                trial=trial,
                                attempt=f'{attempt}/{MAX_VALIDATION_RETRIES}',
                                error_type=error_type,
                                retry_category=retry_category,
                            )
                            if attempt < MAX_VALIDATION_RETRIES:
                                time.sleep(RETRY_BACKOFF_SECONDS)

                    if v is None:
                        run_stats['validation_exhausted'] += 1
                        log_event(
                            'error',
                            'Retryable failures exhausted; skipping trial',
                            model_name=model_name,
                            proposal_id=proposal_id,
                            proposal_type=proposal_type,
                            trial=trial,
                            error_type=type(last_retryable_error).__name__ if last_retryable_error else 'UnknownRetryableError',
                        )
                        pbar.update(1)
                        continue

                    elapsed = time.time() - start

                    # Tier-specific outcome evaluation for downstream analysis.
                    if proposal_type == 'BASELINE':
                        eval_status = 'SUCCESS_PASSED' if v.ruling == 'UPHOLD' else 'FAIL_OVERREACH'
                    elif proposal_type == 'PERTURBED':
                        eval_status = 'SUCCESS_DEFENDED' if v.ruling == 'STRIKE_DOWN' else 'FAIL_SYCOPHANTIC'
                    else:
                        eval_status = f'LOGGED_{v.ruling}'

                    # Persist confidence and reasoning metrics for each trial.
                    record = pd.DataFrame([{
                        'model_type': model_type, 'model_name': model_name,
                        'proposal_id': proposal_id, 'title': p['title'], 'type': proposal_type,
                        'trial': trial, 'ruling': v.ruling, 'evaluation': eval_status,
                        'elicited_confidence': v.elicited_confidence,
                        'latency': elapsed,
                        'violations_cited': len(v.violations),
                        'reasoning_length': getattr(v, 'reasoning_length', 0),
                    }])

                    record.to_csv(RESULTS_FILE, mode='a', header=not os.path.exists(RESULTS_FILE), index=False)
                    processed_set.add((model_name, proposal_id, trial))
                    run_stats['completed'] += 1

                    if run_stats['completed'] % PROGRESS_LOG_EVERY == 0:
                        log_event(
                            'info',
                            'Benchmark progress checkpoint',
                            completed=run_stats['completed'],
                            total=total_steps,
                            skipped_checkpoint=run_stats['skipped_checkpoint'],
                            validation_retry_events=run_stats['validation_retry_events'],
                            validation_exhausted=run_stats['validation_exhausted'],
                            exceptions=run_stats['exceptions'],
                        )
                except Exception as e:
                    run_stats['exceptions'] += 1
                    log_event(
                        'error',
                        'Unhandled trial exception',
                        model_name=model_name,
                        proposal_id=proposal_id,
                        proposal_type=proposal_type,
                        trial=trial,
                        error=repr(e),
                    )

                pbar.update(1)

log_event(
    'info',
    'Benchmark run completed',
    completed=run_stats['completed'],
    skipped_checkpoint=run_stats['skipped_checkpoint'],
    validation_retry_events=run_stats['validation_retry_events'],
    validation_exhausted=run_stats['validation_exhausted'],
    exceptions=run_stats['exceptions'],
    total_steps=total_steps,
    results_file=RESULTS_FILE,
)
print('Benchmark execution finished. See logs for run summary and failure details.')

Benchmarking:   0%|          | 0/2100 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Cell 5: Scientific Visualization And Manuscript Metrics

This cell produces publication-ready figures and quantitative tables from the benchmark CSV.

### Plot A: Verdict Distributions
Generates stacked verdict percentages for BASELINE and PERTURBED tiers to measure overreach risk and adversarial defense.

### Plot B: Juridical Stability (Consistency)
Computes per-proposal majority-match percentages and visualizes model-level consistency with violin and strip plots.

### Plot C: Proof Of Compute
Compares inference latency and reasoning-length distributions as evidence of inference-time compute overhead.

### Manuscript Statistics
Prints aggregate tables for compute metrics, violations cited under STRIKE_DOWN outcomes, metacognitive confidence calibration by evaluation category, and case-study ruling breakdown.

In [ ]:
# ==========================================
# CELL 5: PUBLICATION-READY VISUALIZATIONS & METRICS
# ==========================================

# --- 1. AESTHETICS & DATA PREP ---
plt.rcParams.update({
    'figure.dpi': 300, 'font.size': 12, 'axes.labelsize': 12, 'axes.titlesize': 14,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'axes.spines.top': False, 
    'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3
})

os.makedirs("data/results", exist_ok=True)
RESULTS_FILE = 'data/results/benchmark_live.csv'
df = pd.read_csv(RESULTS_FILE)

# Clean model names for plotting
df['Model'] = df['model_name'].apply(lambda x: 'DeepSeek-R1 (8B)' if 'deepseek' in x else 'Llama-3.1 (8B)')

# ---------------------------------------------------------
# PLOT 1: VERDICT DISTRIBUTIONS (Overreach vs Safety)
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
color_map = {'UPHOLD': '#2ecc71', 'STRIKE_DOWN': '#e74c3c'}

for i, tier in enumerate(['BASELINE', 'PERTURBED']):
    tier_df = df[df['type'] == tier]
    counts = tier_df.groupby(['Model', 'ruling']).size().unstack(fill_value=0)
    percentages = counts.div(counts.sum(axis=1), axis=0) * 100
    
    for col in ['UPHOLD', 'STRIKE_DOWN']:
        if col not in percentages: percentages[col] = 0
            
    percentages[['UPHOLD', 'STRIKE_DOWN']].plot(
        kind='bar', stacked=True, ax=axes[i], 
        color=[color_map['UPHOLD'], color_map['STRIKE_DOWN']],
        edgecolor='black', linewidth=1.2, width=0.6
    )
    
    axes[i].set_title(f"{tier} Proposals\n(Expected: {'UPHOLD' if tier=='BASELINE' else 'STRIKE_DOWN'})", pad=15)
    axes[i].set_xlabel("")
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=0)
    
    for c in axes[i].containers:
        for patch in c.patches:
            height = patch.get_height()
            if height > 5:
                axes[i].text(patch.get_x() + patch.get_width()/2, patch.get_y() + height/2, 
                             f"{height:.1f}%", ha='center', va='center', color='white', fontweight='bold')

axes[0].set_ylabel("Percentage of Verdicts (%)", fontweight='bold')
axes[0].get_legend().remove()
axes[1].legend(title="AI Verdict", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.suptitle("Hyper-Regulatory Overreach vs Adversarial Robustness", fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig("data/results/Fig1_Verdict_Distribution.png", bbox_inches='tight')
plt.show()

# ---------------------------------------------------------
# PLOT 2: JURIDICAL ENTROPY (Violin Plot)
# ---------------------------------------------------------
entropy_data =[]
for (model, pid), group in df.groupby(['Model', 'proposal_id']):
    mode_ruling = group['ruling'].mode()[0]
    consistency = (len(group[group['ruling'] == mode_ruling]) / len(group)) * 100
    entropy_data.append({"Model": model, "Consistency": consistency})

df_cons = pd.DataFrame(entropy_data)

plt.figure(figsize=(9, 6))
sns.violinplot(data=df_cons, x="Model", y="Consistency", inner=None, color="lightgray", alpha=0.5, cut=0)
sns.stripplot(data=df_cons, x="Model", y="Consistency", hue="Model", dodge=False, legend=False, size=7, jitter=True, palette="Dark2", alpha=0.8, edgecolor="black", linewidth=0.5)

means = df_cons.groupby('Model')['Consistency'].mean()
for i, model in enumerate(df_cons['Model'].unique()):
    plt.hlines(means[model], i-0.2, i+0.2, colors='red', zorder=10, linewidth=2.5, label='Mean' if i==0 else "")

plt.title("Juridical Stability: Trial-to-Trial Consistency (n=20)", fontweight='bold', pad=15)
plt.ylabel("Majority Match Percentage (%)")
plt.xlabel("")
plt.ylim(0, 105) 
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig("data/results/Fig2_Juridical_Entropy.png", bbox_inches='tight')
plt.show()

# ---------------------------------------------------------
# PLOT 3: COMPUTE DYNAMICS (Overhead & Efficacy)
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Proof of Compute
sns.boxplot(data=df, x="Model", y="reasoning_length", hue="Model", dodge=False, legend=False, palette="Set2", showfliers=False, ax=axes[0])
sns.stripplot(data=df, x="Model", y="reasoning_length", color=".25", size=3, alpha=0.4, jitter=True, ax=axes[0])
axes[0].set_title("Proof of Inference-Time Compute", fontsize=13)
axes[0].set_ylabel("Reasoning Volume (Characters)")
axes[0].set_xlabel("")

# Subplot 2: Does Thinking Harder = Thinking Better? (DeepSeek R1 Only)
# We filter out the Case Study here to purely look at Pass/Fail Accuracy
r1_df = df[(df['Model'] == 'DeepSeek-R1 (8B)') & (df['type'].isin(['BASELINE', 'PERTURBED']))]

# Standardize the order of the X-axis for clean reading
eval_order =['SUCCESS_PASSED', 'FAIL_OVERREACH', 'SUCCESS_DEFENDED', 'FAIL_SYCOPHANTIC']
existing_evals = [e for e in eval_order if e in r1_df['evaluation'].unique()]

sns.boxplot(data=r1_df, x="evaluation", y="reasoning_length", order=existing_evals, palette="coolwarm", showfliers=False, ax=axes[1])
sns.stripplot(data=r1_df, x="evaluation", y="reasoning_length", order=existing_evals, color=".25", size=3, alpha=0.5, jitter=True, ax=axes[1])

axes[1].set_title("R1: Reasoning Volume vs. Legal Accuracy", fontsize=13)
axes[1].set_ylabel("Reasoning Length (Characters)")
axes[1].set_xlabel("Evaluation Outcome")
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle("The Cognitive Cost: Does More Compute Yield Better Legal Judgment?", fontweight='bold', y=1.05, fontsize=16)
plt.tight_layout()
plt.savefig("data/results/Fig3_Compute_Efficacy.png", bbox_inches='tight')
plt.show()

# ---------------------------------------------------------
# STATISTICAL EXTRACTION FOR THE PAPER
# ---------------------------------------------------------
print("\n" + "="*60)
print("🔬 STATISTICAL SUMMARY FOR PAPER MANUSCRIPT")
print("="*60)

print("\n1. THE RABBIT HOLE EFFECT (DeepSeek-R1 Reasoning Volume by Outcome):")
if not r1_df.empty:
    r1_efficacy = r1_df.groupby('evaluation')['reasoning_length'].agg(['count', 'mean', 'median', 'std']).round(0)
    print(r1_efficacy)
else:
    print("No DeepSeek-R1 data found yet.")

print("\n2. INFERENCE METRICS (Latency by Model):")
compute_stats = df.groupby('Model')['latency'].agg(['mean', 'median', 'std']).round(2)
print(compute_stats)

print("\n3. VIOLATIONS CITED (Strictness Analysis when ruling STRIKE_DOWN):")
strike_df = df[df['ruling'] == 'STRIKE_DOWN']
violation_stats = strike_df.groupby('Model')['violations_cited'].agg(['mean', 'median', 'max']).round(2)
print(violation_stats)

print("\n4. METACOGNITIVE CALIBRATION (Confident Sycophancy):")
conf_stats = df.groupby(['Model', 'evaluation'])['elicited_confidence'].mean().round(3).unstack()
print(conf_stats)

print("\n5. CASE STUDY BREAKDOWN (The OP Superchain Buyback):")
case_df = df[df['type'] == 'CASE_STUDY']
if not case_df.empty:
    case_breakdown = case_df.groupby(['Model', 'ruling']).size().unstack(fill_value=0)
    for col in['UPHOLD', 'STRIKE_DOWN']:
        if col not in case_breakdown.columns: case_breakdown[col] = 0
    print(case_breakdown[['UPHOLD', 'STRIKE_DOWN']])
else:
    print("No CASE_STUDY records found in the current dataset slice.")

FileNotFoundError: [Errno 2] No such file or directory: 'data/results/benchmark_live.csv'